In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



GOLD_DIR = Path("./data/gold")
SILVER_DIR = Path("./data/silver")

# Carga o lectura del DataFrame con Clusters/Riesgo de Módulos anteriores
df_m2 = pd.read_csv(GOLD_DIR / "dataset_riesgo_laft_gold_clustering_modulo2.csv", encoding="utf-8")
df_m2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12121 entries, 0 to 12120
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             12121 non-null  object 
 1   customer_id                12121 non-null  object 
 2   timestamp                  12121 non-null  object 
 3   amount_USD                 12121 non-null  float64
 4   currency_original          12121 non-null  object 
 5   merchant_or_receiver       12121 non-null  object 
 6   payment_description        12060 non-null  object 
 7   customer_country           12121 non-null  object 
 8   customer_activity          12121 non-null  object 
 9   channel                    12121 non-null  object 
 10  accumulated_amount_window  12121 non-null  float64
 11  tx_count_window            12121 non-null  int64  
 12  is_smurfing                12121 non-null  bool   
 13  fuera_de_clustering        12121 non-null  boo

   ### Clase para analisis de NLP

In [2]:
class PaymentNLPAnalyzer:
    """Pipeline modular de NLP para detección de riesgo semántico en descripciones de pago (LAFT / Fraude)."""

    def __init__(
        self,
        high_risk_keywords: list = None,
        low_risk_keywords: list = None,
        max_features: int = 100,
    ):
        # Lexicones contextuales para LAFT (Flexibles / Parámetros por defecto)
        self.high_risk_keywords = high_risk_keywords or [
            # Términos atípicos/estructurados o transfronterizos observados en el dataset
            "estructurada",
            "estructurado",
            "lote",
            "frontera",
            "crypto",
            "inversion crypto",
            "internacional",
            # Términos generales de alto riesgo LAFT
            "efectivo",
            "urgente",
            "sin factura",
            "fraccionado",
            "prestamo",
            "comision",
            "moneda extranjera",
            "cambio",
            "honorarios no registrados",
            "donacion libre",
            "pago directo",
            "terceros",
            "supermercado",
            "restaurante",
            "cortinas",
            "familiar",
            "suscripcion",
        ]

        self.low_risk_keywords = low_risk_keywords or [
            # Términos de transacciones habituales/operativas observadas en el dataset
            "factura",
            "servicios",
            "segura",
            # Términos generales de bajo riesgo LAFT
            "nomina",
            "arriendo",
            "educacion",
            "servicios publicos",
            "compra",
            "mantenimiento",
            "poliza",
        ]

        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2), max_features=max_features
        )
        self.is_fitted = False

    def preprocess_text(self, text: str) -> str:
        """Limpia y normaliza el texto de la descripción de pago."""
        if not isinstance(text, str) or pd.isna(text):
            return ""

        text = text.lower()
        # Eliminar caracteres especiales y números aislados
        text = re.sub(r"[^\w\s]", " ", text)
        text = re.sub(r"\d+", " ", text)
        # Reemplazar múltiples espacios
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def fit(self, raw_texts: pd.Series):
        """Entrena la representación TF-IDF con las descripciones del corpus."""
        clean_texts = raw_texts.apply(self.preprocess_text)
        self.vectorizer.fit(clean_texts)
        self.is_fitted = True
        return self

    def _calculate_lexicon_matches(self, clean_text: str) -> float:
        """Calcula el sesgo numérico basado en palabras clave de alto vs bajo riesgo."""
        risk_hits = sum(
            1 for kw in self.high_risk_keywords if kw in clean_text
        )
        legit_hits = sum(1 for kw in self.low_risk_keywords if kw in clean_text)

        # Retorna un valor neto (-1 a +1 relativo)
        if risk_hits == 0 and legit_hits == 0:
            return 0.0
        return (risk_hits - legit_hits) / (risk_hits + legit_hits + 1e-5)

    def calculate_nlp_scores(self, df: pd.DataFrame, text_col: str = "payment_description") -> pd.DataFrame:
        """
        Genera métricas semánticas y la columna final 'nlp_risk_score'.
        Escala: [0.0 = Bonificación máxima / Muy Legítimo, 1.0 = Penalización máxima / Alto Riesgo]
        """
        if not self.is_fitted:
            self.fit(df[text_col])

        clean_texts = df[text_col].apply(self.preprocess_text)

        # Matriz TF-IDF
        tfidf_matrix = self.vectorizer.transform(clean_texts)

        # Similitud con perfiles de referencia (Centros de Gravedad Semánticos)
        high_risk_doc = " ".join(self.high_risk_keywords)
        low_risk_doc = " ".join(self.low_risk_keywords)

        ref_matrix = self.vectorizer.transform([high_risk_doc, low_risk_doc])

        sim_high_risk = cosine_similarity(tfidf_matrix, ref_matrix[0:1]).flatten()
        sim_low_risk = cosine_similarity(tfidf_matrix, ref_matrix[1:2]).flatten()

        # Puntuación basada en coincidencias directas
        lexicon_bias = clean_texts.apply(self._calculate_lexicon_matches)

        # Ensamble de Score Semántico (Rango continuo de 0 a 1)
        # Score base neutro = 0.50
        raw_score = 0.50 + (sim_high_risk * 0.35) - (sim_low_risk * 0.35) + (lexicon_bias * 0.15)
        nlp_risk_score = np.clip(raw_score, 0.0, 1.0)

        # Generación de Flag Contextual (Descuento o Penalización)
        context_flag = np.where(
            nlp_risk_score > 0.65,
            "Penalización Semántica (Riesgo)",
            np.where(nlp_risk_score < 0.35, "Bonificación Semántica (Legítimo)", "Contexto Neutro")
        )

        df_out = df.copy()
        df_out["clean_payment_description"] = clean_texts
        df_out["nlp_risk_score"] = np.round(nlp_risk_score, 4)
        df_out["context_nlp_flag"] = context_flag

        return df_out

In [3]:


# Instanciar el analizador modular de NLP
nlp_analyzer = PaymentNLPAnalyzer()

# Ajustar el modelo TF-IDF y calcular el nlp_risk_score
df_modulo3 = nlp_analyzer.calculate_nlp_scores(
    df=df_m2, 
    text_col="payment_description"
)

# Verificación de resultados
cols_interes = [
    "transaction_id", 
    "payment_description", 
    "nivel_riesgo", 
    "nlp_risk_score", 
    "context_nlp_flag"
]
print("=== RESULTADOS MÓDULO III: ANÁLISIS SEMÁNTICO (NLP) ===")
df_modulo3[cols_interes].sample(10)

=== RESULTADOS MÓDULO III: ANÁLISIS SEMÁNTICO (NLP) ===


,transaction_id,payment_description,nivel_riesgo,nlp_risk_score,context_nlp_flag
4742,TRX_000758,Compra supermercado,Riesgo bajo,0.4742,Contexto Neutro
11176,TRX_003756,Suscripción mensual,Riesgo medio,0.5000,Contexto Neutro
6660,TRX_002073,Restaurante,Riesgo bajo,0.6820,Penalización Semántica (Riesgo)
11195,TRX_006660,Restaurante,Riesgo bajo,0.6820,Penalización Semántica (Riesgo)
2525,TRX_011666,Pago servicios,Riesgo bajo,0.2789,Bonificación Semántica (Legítimo)
5451,TRX_006470,Suscripción mensual,Riesgo bajo,0.5000,Contexto Neutro
8951,TRX_006705,Suscripción mensual,Riesgo bajo,0.5000,Contexto Neutro
9549,TRX_002235,Compra supermercado,Riesgo bajo,0.4742,Contexto Neutro
4055,TRX_002712,Pago servicios,Riesgo medio,0.2789,Bonificación Semántica (Legítimo)
5917,TRX_005811,Pago servicios,Riesgo bajo,0.2789,Bonificación Semántica (Legítimo)


In [4]:
df_modulo3.sample(5)

,transaction_id,customer_id,timestamp,amount_USD,currency_original,merchant_or_receiver,payment_description,customer_country,customer_activity,channel,accumulated_amount_window,tx_count_window,is_smurfing,fuera_de_clustering,cluster_id,nivel_riesgo,clean_payment_description,nlp_risk_score,context_nlp_flag
637,TRX_001275,0d848,2025-01-10 06:18:00,12.94,COP,REC_5122,Restaurante,CO,Independiente,BRANCH,12.94,1,False,False,0,Riesgo bajo,restaurante,0.6820,Penalización Semántica (Riesgo)
1013,TRX_009684,137af,2025-03-12 21:19:00,66.77,MXN,REC_8944,Restaurante,MX,Comerciante,APP,66.77,1,False,False,0,Riesgo bajo,restaurante,0.6820,Penalización Semántica (Riesgo)
10455,TRX_008405,df7b1,2025-03-03 16:09:00,93.90,COP,REC_1914,Suscripción mensual,CO,Independiente,BRANCH,93.90,1,False,False,0,Riesgo bajo,suscripción mensual,0.5000,Contexto Neutro
7179,TRX_008870,9ab97,2025-03-07 03:31:00,26.91,MXN,REC_7226,Transferencia familiar,MX,Asalariado,WEB,26.91,1,False,False,0,Riesgo bajo,transferencia familiar,0.6684,Penalización Semántica (Riesgo)
7159,TRX_008172,9a986,2025-03-01 22:51:00,32.03,COP,REC_1860,Transferencia familiar,CO,Comerciante,APP,32.03,1,False,False,0,Riesgo bajo,transferencia familiar,0.6684,Penalización Semántica (Riesgo)


In [5]:
# Guardar en la capa gold
out_path = GOLD_DIR / "dataset_riesgo_laft_gold_NLP_riskscore_modulo3.csv"
df_modulo3.to_csv(out_path, index=False, encoding="utf-8")
print("\nGuardado en:", out_path.resolve())


Guardado en: C:\Users\Hugo\OneDrive\Documentos\MEGA\PROYECTOS\GIT-HUB\reto_analista_de_datos\data\gold\dataset_riesgo_laft_gold_NLP_riskscore_modulo3.csv


In [6]:
# clientes de alto riesgo
clientres_riesgosos = df_modulo3[df_modulo3['nivel_riesgo'] == 'Riesgo alto']
clientres_riesgosos.sample(10)

,transaction_id,customer_id,timestamp,amount_USD,currency_original,merchant_or_receiver,payment_description,customer_country,customer_activity,channel,accumulated_amount_window,tx_count_window,is_smurfing,fuera_de_clustering,cluster_id,nivel_riesgo,clean_payment_description,nlp_risk_score,context_nlp_flag
8578,TRX_007762,b5f51,2025-02-27 00:16:00,206.29,USD,REC_STRUCTURING,Pago factura,PA,Comerciante,APP,1207.42,5,True,False,1,Riesgo alto,pago factura,0.2868,Bonificación Semántica (Legítimo)
8505,TRX_009010,b4957,2025-03-08 01:59:00,198.13,MXN,REC_7299,Compra supermercado,MX,Asalariado,APP,198.13,1,False,False,1,Riesgo alto,compra supermercado,0.4742,Contexto Neutro
6342,TRX_003390,87be5,2025-01-26 00:22:00,480.71,USD,REC_STRUCTURING,Pago factura,CO,Comerciante,APP,2397.83,5,True,False,1,Riesgo alto,pago factura,0.2868,Bonificación Semántica (Legítimo)
3696,TRX_004203,4c93d,2025-02-01 00:42:00,236.88,USD,REC_STRUCTURING,Pago factura,MX,Comerciante,APP,2891.31,12,True,False,1,Riesgo alto,pago factura,0.2868,Bonificación Semántica (Legítimo)
8484,TRX_005875,b4957,2025-02-12 17:02:00,117.41,MXN,REC_9759,Compra supermercado,MX,Asalariado,APP,117.41,1,False,False,1,Riesgo alto,compra supermercado,0.4742,Contexto Neutro
8589,TRX_008546,b5f51,2025-03-04 16:07:00,70.77,COP,REC_9582,Compra supermercado,CO,Asalariado,WEB,70.77,1,False,False,1,Riesgo alto,compra supermercado,0.4742,Contexto Neutro
3694,TRX_004201,4c93d,2025-02-01 00:40:00,241.76,USD,REC_STRUCTURING,Pago factura,CO,Comerciante,APP,2446.23,10,True,False,1,Riesgo alto,pago factura,0.2868,Bonificación Semántica (Legítimo)
3690,TRX_004196,4c93d,2025-02-01 00:25:00,252.56,USD,REC_STRUCTURING,Pago factura,PA,Comerciante,APP,1483.53,6,True,False,1,Riesgo alto,pago factura,0.2868,Bonificación Semántica (Legítimo)
8481,TRX_003549,b4957,2025-01-27 02:36:00,100.37,MXN,REC_2783,Suscripción mensual,MX,Asalariado,APP,100.37,1,False,False,1,Riesgo alto,suscripción mensual,0.5000,Contexto Neutro
3695,TRX_004202,4c93d,2025-02-01 00:40:00,208.20,USD,REC_STRUCTURING,Pago factura,MX,Comerciante,APP,2654.43,11,True,False,1,Riesgo alto,pago factura,0.2868,Bonificación Semántica (Legítimo)


In [7]:
df_modulo3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12121 entries, 0 to 12120
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             12121 non-null  object 
 1   customer_id                12121 non-null  object 
 2   timestamp                  12121 non-null  object 
 3   amount_USD                 12121 non-null  float64
 4   currency_original          12121 non-null  object 
 5   merchant_or_receiver       12121 non-null  object 
 6   payment_description        12060 non-null  object 
 7   customer_country           12121 non-null  object 
 8   customer_activity          12121 non-null  object 
 9   channel                    12121 non-null  object 
 10  accumulated_amount_window  12121 non-null  float64
 11  tx_count_window            12121 non-null  int64  
 12  is_smurfing                12121 non-null  bool   
 13  fuera_de_clustering        12121 non-null  boo